<a href="https://colab.research.google.com/github/Muslinmin/special-octo-barnacle/blob/main/FetchReach_SAC_2201360_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# -*- coding: utf-8 -*-
"""FetchReach_SAC.ipynb

Train FetchReach-v4 using SAC (Stable-Baselines3)
"""

# ============================================================
# 1. Install Dependencies
# ============================================================
!pip install stable-baselines3 gymnasium-robotics mujoco

import numpy as np
import gymnasium as gym
import gymnasium_robotics
import os

os.environ['MUJOCO_GL'] = 'egl'

from stable_baselines3 import SAC
from stable_baselines3.common.evaluation import evaluate_policy


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.5/187.5 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.2/26.2 MB 38.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 49.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.5/852.5 kB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 243.5/243.5 kB 8.8 MB/s eta 0:00:00


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [2]:
# ============================================================
# 2. Create Environment
# ============================================================
env = gym.make("FetchReach-v4", reward_type="dense", render_mode="rgb_array")

print("Observation space:", env.observation_space)
print("Action space:", env.action_space)

Observation space: Dict('achieved_goal': Box(-inf, inf, (3,), float64), 'desired_goal': Box(-inf, inf, (3,), float64), 'observation': Box(-inf, inf, (10,), float64))
Action space: Box(-1.0, 1.0, (4,), float32)


In [3]:
# ============================================================
# 3. Train SAC
# ============================================================
total_timesteps = 50_000

model = SAC(
    "MultiInputPolicy",
    env,
    verbose=1,
    gamma=0.99,
    learning_rate=0.001,
    batch_size=256,
    learning_starts=1000,
)

model.learn(total_timesteps=total_timesteps)

model.save("fetchreach_sac_model")


Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 50       |
|    ep_rew_mean     | -11.5    |
|    success_rate    | 0        |
| time/              |          |
|    episodes        | 4        |
|    fps             | 342      |
|    time_elapsed    | 0        |
|    total_timesteps | 200      |
---------------------------------


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


---------------------------------
| rollout/           |          |
|    ep_len_mean     | 50       |
|    ep_rew_mean     | -11.4    |
|    success_rate    | 0        |
| time/              |          |
|    episodes        | 8        |
|    fps             | 341      |
|    time_elapsed    | 1        |
|    total_timesteps | 400      |
---------------------------------
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 50       |
|    ep_rew_mean     | -11      |
|    success_rate    | 0        |
| time/              |          |
|    episodes        | 12       |
|    fps             | 341      |
|    time_elapsed    | 1        |
|    total_timesteps | 600      |
---------------------------------
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 50       |
|    ep_rew_mean     | -10.7    |
|    success_rate    | 0        |
| time/              |          |
|    episodes        | 16       |
|    fps      

In [4]:
# ============================================================
# 4. Evaluate with evaluate_policy (100 episodes)
# ============================================================
eval_env = gym.make("FetchReach-v4", reward_type="dense", render_mode="rgb_array")

mean_reward, std_reward = evaluate_policy(model, eval_env, n_eval_episodes=100)
print(f"\n[evaluate_policy] Mean reward: {mean_reward:.2f} +/- {std_reward:.2f}")

/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/evaluation.py:71: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(



[evaluate_policy] Mean reward: -0.88 +/- 0.27


In [5]:
# ============================================================
# 5. Manual Evaluation (100 episodes)
# ============================================================
manual_rewards = []

for ep in range(100):
    obs, info = eval_env.reset()
    ep_reward = 0.0
    done = False

    while not done:
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = eval_env.step(action)
        ep_reward += reward
        done = terminated or truncated

    manual_rewards.append(ep_reward)

manual_mean = np.mean(manual_rewards)
manual_std = np.std(manual_rewards)
print(f"\n[Manual Eval] Mean reward: {manual_mean:.2f} +/- {manual_std:.2f}")

print("\n--- Comparison ---")
print(f"evaluate_policy : {mean_reward:.2f} +/- {std_reward:.2f}")
print(f"Manual eval     : {manual_mean:.2f} +/- {manual_std:.2f}")
print("Both should be very close since evaluate_policy does the same thing internally.")

eval_env.close()



[Manual Eval] Mean reward: -0.92 +/- 0.28

--- Comparison ---
evaluate_policy : -0.88 +/- 0.27
Manual eval     : -0.92 +/- 0.28
Both should be very close since evaluate_policy does the same thing internally.


In [6]:
# ============================================================
# 6. Video Recording + Distance / Success Tracking
# ============================================================
!apt-get install -y xvfb

from gymnasium.wrappers import RecordVideo
import glob
from IPython.display import Video

vid_env = gym.make("FetchReach-v4", reward_type="dense", render_mode="rgb_array")
vid_env = RecordVideo(vid_env, video_folder="video_sac",
                      episode_trigger=lambda x: True)

obs, info = vid_env.reset()
goal_pos = obs["desired_goal"]
effector_pos = obs["observation"][:3]
print(f"Start effector pos: {effector_pos}")
print(f"Goal pos: {goal_pos}")

total_reward = 0.0
total_euclidean = 0.0
min_dist = float('inf')
success_achieved = False

for step in range(50):
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = vid_env.step(action)

    effector_pos = obs["observation"][:3]
    goal_pos = obs["desired_goal"]
    dist = np.linalg.norm(effector_pos - goal_pos)
    total_reward += reward
    total_euclidean += dist

    if dist < min_dist:
        min_dist = dist
    if info.get("is_success", 0.0):
        success_achieved = True

    print(f"Step {step+1:2d} | Pos: {effector_pos} | Dist: {dist:.4f} | Reward: {reward:.4f} | Success: {info.get('is_success', 0.0)}")

    if terminated or truncated:
        break

vid_env.close()

print(f"\n--- Trajectory Summary ---")
print(f"Total reward (sum): {total_reward:.4f}")
print(f"Negative total Euclidean distance: {-total_euclidean:.4f}")
print(f"These should be equal (dense reward = -dist at each step).")
print(f"Min distance achieved: {min_dist:.4f}")
print(f"Success achieved (dist <= 0.05): {success_achieved}")

# Play video
mp4list = glob.glob('video_sac/*.mp4')
mp4list.sort()
if mp4list:
    print(f"Video: {mp4list[-1]}")
    Video(mp4list[-1], embed=True)

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
xvfb is already the newest version (2:21.1.4-2ubuntu1.7~22.04.16).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.
Start effector pos: [1.34183475 0.74910103 0.53472518]
Goal pos: [1.37513187 0.8886412  0.59777314]
Step  1 | Pos: [1.36415054 0.78009911 0.56475073] | Dist: 0.1140 | Reward: -0.1140 | Success: 0.0
Step  2 | Pos: [1.37327348 0.81336795 0.59483174] | Dist: 0.0754 | Reward: -0.0754 | Success: 0.0
Step  3 | Pos: [1.37486953 0.84152841 0.59988183] | Dist: 0.0472 | Reward: -0.0472 | Success: 1.0
Step  4 | Pos: [1.37312297 0.85388598 0.59964267] | Dist: 0.0349 | Reward: -0.0349 | Success: 1.0
Step  5 | Pos: [1.37142028 0.86134304 0.59979924] | Dist: 0.0276 | Reward: -0.0276 | Success: 1.0
Step  6 | Pos: [1.36986552 0.86699474 0.59974323] | Dist: 0.0224 | Reward: -0.0224 | Success: 1.0
Step  7 | Pos: [1.36848988 0.8713482  0.59945408] | Dist: 0.0186 | Reward: -0.0186 | 

/usr/local/lib/python3.12/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)



--- Trajectory Summary ---
Total reward (sum): -0.8817
Negative total Euclidean distance: -0.8817
These should be equal (dense reward = -dist at each step).
Min distance achieved: 0.0124
Success achieved (dist <= 0.05): True
Video: video_sac/rl-video-episode-0.mp4
